# Relationships: 1:N, M:N, 1:1

In the previous notebook you created foreign keys. Now we will look at the
**three types of relationships** that can exist between tables:

- **1:N** — One-to-Many
- **M:N** — Many-to-Many
- **1:1** — One-to-One

## Part A: One-to-Many Relationships (1:N)

### What 1:N means

A **1:N** relationship means:
- **One** record in the first table can be associated with **many** records
  in the second
- **Every** record in the second table is associated with **exactly one**
  record in the first

### Everyday examples

- **One** mother can have **many** children. Each child has **one** biological
  mother. → 1:N
- **One** professor teaches **many** courses. Each course has **one** main
  professor. → 1:N
- **One** country has **many** cities. Each city belongs to **one** country.
  → 1:N

### Example from Chinook

In Chinook, **one** artist may have released **many** albums. Each album,
however, belongs to **exactly one** artist.

So: `artist (1) ←——— album (N)`

In [ ]:
from IPython.display import SVG
SVG(filename="../../pgmodeler/exports/album_artist_with_fk_highlighted.svg")

### Where does the FK live in a 1:N relationship?

Always on the **«many» side**. In our example, the `album` table (the «many»
side) has the `artist_id` column as a foreign key pointing to `artist` (the
«one» side).

If it were the other way around — that is, if we tried to put the FK in
`artist` pointing to `album` — `artist` would have a single `album_id` column.
But an artist has **many** albums and a column can hold **only one** value.
This is why the FK always goes on the «many» side.

### Exact cardinalities in pgModeler

At the ends of the line, pgModeler shows the **exact cardinalities**.

For the `artist ←——— album` relationship:

| End | Notation | Meaning |
|---|---|---|
| **artist** side | `1..1` | Every album references **exactly one** artist — mandatory |
| **album** side | `0..N` | Every artist can have **from zero to many** albums |

The notation **`0..N`** reads as «minimum 0, maximum N». It means it is
perfectly acceptable for an artist to have no albums on record.

The label «**1:N**» describes the **category** of the relationship. The exact
cardinalities (`1..1`, `0..N`, `1..N`) are **specialisations** within that
category.

### The 1:N relationships in your project

Open pgModeler and the file
`student_work/my_project/stage_1_initial_design.dbm`. Observe the two lines
you created:

**Relationship 1: book ← loan**
- On the **book** side: `1..1` (every loan must refer to exactly one book)
- On the **loan** side: `0..N` (every book can have zero or many loan records)

**Relationship 2: member ← loan**
- On the **member** side: `1..1`
- On the **loan** side: `0..N`

In both relationships the FK lives on the «many» side — in the `loan` table.

> The `loan` table has **two** incoming 1:N relationships. This is very
> common: tables like `loan` act as **junction tables** between other tables.

## Part B: Many-to-Many Relationships (M:N)

### What M:N means

A **M:N** relationship means:
- **Many** records in the first table can be associated with **many** records
  in the second, and vice versa

### Examples

- **One** student enrols in **many** courses. **One** course has **many**
  students. → M:N
- **One** book can have **many** authors. **One** author can have written
  **many** books. → M:N
- **One** actor appears in **many** films. **One** film has **many** actors.
  → M:N

### Example from Chinook

In Chinook, **one** track can belong to **many** playlists. **One** playlist
contains **many** tracks.

So: `track (M) ←———→ playlist (N)`

In [ ]:
SVG(filename="../../pgmodeler/exports/many_to_many_track_playlist.svg")

### The solution: junction table

Notice that there is **no direct line** between `track` and `playlist`.
Instead, there is a **third table**, `playlist_track`, which:

- Has an FK to `playlist`
- Has an FK to `track`

Each record in `playlist_track` represents **one link** between a specific
track and a specific playlist.

### Why is a junction table necessary?

In relational databases, **we cannot** define a direct M:N connection. An FK
column holds **one value**, so:

- We cannot put an FK in `track` pointing to **many** playlists at once
- Nor an FK in `playlist` pointing to **many** tracks

The solution: **break** the M:N relationship into **two 1:N relationships**
via a junction table (`playlist_track`). This gives us:

- `playlist (1) ←——— playlist_track (N)`
- `track (1) ←——— playlist_track (N)`

Two 1:N relationships that together implement the M:N relationship.

### M:N in your own project

In your project, notice two columns of the `book` table:

- `authors` — stores the authors of a book as a comma-separated text string
  (e.g. `"Tolkien, Lewis"`)
- `genres` — stores the genres as text as well (e.g. `"Fantasy, Adventure"`)

This design hides two M:N relationships:
- A book can have **many** authors. An author can have written **many** books.
  → M:N
- A book can belong to **many** genres. A genre has **many** books. → M:N

> **Important**: At this stage of the project, **we keep this design as it
> is** — that is, as comma-separated text, not as separate tables. In
> **module 05** we will see why this approach causes problems and how to fix
> it.

## Part C: One-to-One Relationships (1:1)

### What 1:1 means

A **1:1** relationship means:
- **One** record in the first table is associated with **at most one** record
  in the second
- And vice versa

### Everyday examples

- Every **citizen** has **one** unique tax identification number. Every tax ID
  belongs to **one** citizen. → 1:1
- Every **employee** has **one** personal email account at the company. → 1:1

### When is it used?

1:1 is the **rarest** relationship. It is used mainly when:
- We want to **split** a large table into two for organisational reasons
- We have **optional** data that does not apply to all records
- We want to store **sensitive data** in a separate table with more restricted
  access rights

### How is it implemented?

As with 1:N, we place **an FK in one of the two tables**. The difference: the
FK must also have a **UNIQUE constraint**, so that each value appears **only
once**. This ensures that the same record cannot be linked to more than one
record in the other table.

### Practice: add a 1:1 relationship to your project

It is time to apply the theory. Your library will manage **member cards**:
each member has **one** personal card, and each card belongs to **one** member.

#### Step 1: Open your model

If it is not already open: pgModeler → **File → Open** →
`student_work/my_project/stage_1_initial_design.dbm`

#### Step 2: Create the member_card table

Using **New Table**, create a new table `member_card` with the following
columns:

| Name | Type | Note |
|---|---|---|
| `card_id` | integer | **Primary Key** |
| `member_id` | integer | Will become FK with UNIQUE constraint |
| `issue_date` | date | When the card was issued |
| `expiry_date` | date | When it expires |

Don't forget to **set `card_id` as PK** (as we learned in notebook 02).

#### Step 3: Add FK with UNIQUE constraint

Now you will connect `member_card` to `member` in a way that enforces a 1:1
relationship.

We could do this as in the previous notebook by adding a **Constraint** to
`member_id` but instead of **Foreign Key** we choose **Unique**. However,
pgModeler offers an even easier and more practical solution:

1. Right-click the `member_card` table
2. **New → Relationship → One to One**
3. The connection line appears automatically. Move over to `member` and select
   `member_id`
4. A window opens with a name already suggested for the relationship. Under
   cardinality, check **member is required** and click **Apply**
5. The relationship between the tables has been created

> **Note**: A new column `member_id_member` is created automatically as an FK,
> along with two Constraints at the bottom one for the FK and one for the
> Unique constraint. The `member_id` column you added manually to `member_card`
> earlier is therefore redundant and can be deleted.

#### Step 5: Save

Click **File → Save** (Ctrl+S).

#### How to confirm it is 1:1

Observe the new connecting line between `member_card` and `member`. The
cardinalities should show:

- On the **member** side: `1..1`
- On the **member_card** side: `0..1` or `1..1` (instead of `0..N`)

The **maximum of 1** on the `member_card` side is what turns the relationship
from 1:N into 1:1, thanks to the UNIQUE constraint we added.

You can also browse the Chinook schema and inspect the relationships between
its tables.

## Key takeaways

- There are **three types of relationships**: 1:N, M:N, 1:1
- In **1:N** relationships the FK always lives on the «many» side
- In **M:N** relationships a **junction table** is needed to implement the
  relationship
- In **1:1** relationships the FK requires a **UNIQUE constraint**
- In pgModeler you recognise relationships by the **cardinality symbols** at
  the ends of the lines (`1..1`, `0..N`, etc.)
- Your project now has **4 tables**: book, member, loan, member_card

In the next notebook we will look at the **data types** used for columns.